# EconoCausal — Week 1 EDA Notebook
**Project reference:** ITS/DSML/1147

Runs the full EDA for the synthetic retail campaign dataset.

> **Data notice:** All data is entirely synthetic — no real customers.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').resolve().parent))

import pandas as pd
import matplotlib
matplotlib.use('inline')
from IPython.display import Image, display
print('Dependencies loaded.')

## 1. Load Cleaned Dataset

In [ ]:
df = pd.read_csv('../data/processed/cleaned_retail_campaign.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Descriptive Statistics

In [ ]:
df.describe(include='all')

## 3. Missing Values and Duplicates

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
outlier_count = int(df['is_outlier'].sum())
outlier_pct = 100 * df['is_outlier'].mean()
print(f'Outlier-flagged rows: {outlier_count} ({outlier_pct:.1f}%)')

## 4. Treatment and Control Group Sizes

In [ ]:
display(Image('../reports/figures/01_treatment_control_counts.png'))
print(df['discount_offered'].value_counts())

## 5. Purchase Rate: Naïve Comparison

> The gap shown here is **not** the causal effect of the discount — it conflates the discount effect with confounding from pre-treatment variables.

In [ ]:
display(Image('../reports/figures/02_purchase_rate_by_treatment.png'))
treated_rate = df.loc[df['discount_offered']==1, 'purchased'].mean()
control_rate  = df.loc[df['discount_offered']==0, 'purchased'].mean()
naive_ate = treated_rate - control_rate
print(f'Treated purchase rate : {treated_rate:.1%}')
print(f'Control purchase rate : {control_rate:.1%}')
print(f'Naïve ATE (biased)    : {naive_ate:+.1%}')
print(f'True embedded ATE     : ~+8.0%  (by design in the DGP)')

## 6. Numeric Feature Distributions

In [ ]:
display(Image('../reports/figures/03_numeric_distributions.png'))

## 7. Categorical Feature Distributions

In [ ]:
display(Image('../reports/figures/04_categorical_distributions.png'))

## 8. Correlation Heatmap

> Correlation is a descriptive tool only and does not imply causation.

In [ ]:
display(Image('../reports/figures/05_correlation_heatmap.png'))

## 9. Purchase Rate by Segment and Age Group

In [ ]:
display(Image('../reports/figures/06_purchase_rate_by_segment.png'))
display(Image('../reports/figures/07_purchase_rate_by_age_group.png'))

## 10. Confounder Evidence

In [ ]:
display(Image('../reports/figures/08_propensity_by_confounder.png'))

## 11. Why the Naïve Estimate Is Biased

Treatment was **not randomly assigned**. Customers with more `past_purchases`,
higher `website_visits`, and Premium/Regular segments had higher treatment
propensity AND higher baseline purchase probability.

**Double Machine Learning (Week 2)** removes this bias by:
1. Partialling out confounder effects from both outcome and treatment.
2. Estimating the treatment effect from the residualised relationship.

## 12. Summary

In [ ]:
summary = {
    'n_total'         : len(df),
    'n_treated'       : int((df['discount_offered']==1).sum()),
    'n_control'       : int((df['discount_offered']==0).sum()),
    'naive_ate'       : round(float(naive_ate), 4),
    'true_ate_approx' : 0.08,
    'confounders'     : ['past_purchases','avg_order_value',
                         'website_visits','customer_segment','age_group'],
}
for k, v in summary.items():
    print(f'{k:28s}: {v}')